# TP 2 — NumPy : l'image comme tableau

**Objectifs**

- Passer d'une image Pillow à un `ndarray` NumPy et inversement.
- Inspecter `shape`, `dtype`, dynamique d'intensité.
- Manipuler les canaux séparément.
- Mettre en œuvre les opérations ponctuelles usuelles : négatif, seuillage, niveaux de gris pondéré.
- Construire une image synthétique (gradient, échiquier) en partant d'un tableau vide.
- Assembler une mosaïque 2×2 combinant plusieurs traitements d'une même image.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage import data

img = Image.fromarray(data.astronaut())  # photo RGB 512x512
arr = np.array(img)
print(arr.shape, arr.dtype, arr.min(), arr.max())

(512, 512, 3) uint8 0 255


## Exercice 1 — Inspection et canaux

1. Affichez la `shape`, le `dtype`, ainsi que `min`, `max` et `mean` du tableau `arr`.
2. Extrayez les trois canaux `R`, `G`, `B` dans trois tableaux 2D séparés.
3. Affichez les trois canaux côte à côte en niveaux de gris (chacun représente l'intensité du canal correspondant).

<details>
<summary>💡 Coup de pouce — extraire les 3 canaux RGB</summary>

**🎯 But pédagogique :** comprendre qu'une image RGB en NumPy est un tableau **3D** de shape `(H, W, 3)`, et que chaque canal est juste une **« tranche »** sur le dernier axe.

**Extraction des canaux**

Le `...` (ellipsis) signifie « toutes les autres dimensions, on s'en occupe pas » :

```python
r = arr[..., 0]   # équivalent à arr[:, :, 0] → tableau 2D (H, W)
g = arr[..., 1]
b = arr[..., 2]
```

Chaque canal sorti est **2D** : c'est une carte d'intensité d'un seul canal. À l'affichage, on doit dire à Matplotlib que c'est du gris (sinon il applique sa colormap par défaut `viridis` et ça ressort en jaune/violet) :

```python
ax.imshow(r, cmap="gray", vmin=0, vmax=255)
```

⚠️ **Toujours mettre `vmin=0, vmax=255`** quand on compare plusieurs canaux côte à côte. Sans ça, Matplotlib recale automatiquement l'échelle de chaque sous-figure indépendamment → un canal sombre paraîtra clair par recadrage, comparaison faussée.

**Astuce visuelle :** sur la photo `coffee()`, le canal **rouge** sera très clair (la tasse rouge), le canal **bleu** très sombre (peu de bleu dans l'image). C'est la preuve que les canaux capturent bien des informations différentes.

</details>

In [2]:
# TODO

## Exercice 2 — Région d'intérêt et écriture

1. Découpez une région d'intérêt (`roi`) correspondant aux lignes 100 à 300 et colonnes 200 à 400 ; affichez-la.
2. Sur une **copie** de `arr`, mettez à zéro le canal vert ; affichez le résultat.
3. Sur une autre copie, peignez un rectangle rouge plein (255, 0, 0) entre les lignes 50–100 et les colonnes 50–450.

> Pourquoi `arr.copy()` ? Parce que `np.asarray(img)` peut renvoyer une vue en lecture seule. Travailler sur une copie évite les surprises.

<details>
<summary>💡 Coup de pouce — slicing et écriture dans une image</summary>

**🎯 But :** manipuler des **régions** et des **canaux** directement avec la syntaxe NumPy. Pas de boucles Python — tout par slicing.

**Sélection de région (ROI)**

Le slicing classique `[lignes, colonnes]`. Attention : l'ordre est **(H, W)**, pas (x, y) comme en géométrie classique :

```python
roi = arr[100:300, 200:400]    # lignes 100→299, colonnes 200→399
```

C'est l'inverse de Pillow où `img.size = (largeur, hauteur)` : avec NumPy, **le premier axe est toujours la hauteur**. Erreur classique des débutants.

**Annuler un canal**

`...` désigne tous les axes restants. Pour mettre le canal vert à 0 sur une copie :

```python
cp = arr.copy()        # important : sinon on modifie l'original !
cp[..., 1] = 0         # tout le canal vert, partout, à 0
```

**Dessiner un rectangle plein**

NumPy **broadcaste** automatiquement un tuple de 3 valeurs sur les 3 canaux :

```python
cp[50:100, 50:450] = (255, 0, 0)   # rouge pur dans la région ; G et B à 0
```

⚠️ **Toujours `arr.copy()` avant modification** si on veut garder l'original intact. Sinon NumPy modifie en place et l'original est perdu (sous-tableaux en `view`, pas en `copy`).

</details>

In [3]:
# TODO

## Exercice 3 — Opérations ponctuelles

Calculez et affichez :

1. Le **négatif** de l'image (`255 - arr`).
2. Une version en **niveaux de gris pondérée** par les coefficients de luminance perceptuelle `[0.299, 0.587, 0.114]`. Vérifiez le dtype et la dynamique du résultat.
3. Une version **binarisée** par seuillage à 128 sur l'image en niveaux de gris.

Affichez les trois résultats sur une figure 1×3.

<details>
<summary>💡 Coup de pouce — opérations ponctuelles sur les pixels</summary>

**🎯 But :** transformer la valeur de chaque pixel selon une formule simple, sans boucle.

**Négatif photographique**

```python
neg = 255 - arr
```

Inverse l'intensité. Fonctionne pour des images uint8 (et NumPy gère le broadcast canal par canal). Pour des images float [0, 1], écrire `1.0 - arr`.

**Luminance pondérée (RGB → gris)**

La formule standard ITU-R BT.601 :

```python
gray = arr @ [0.299, 0.587, 0.114]
```

Le `@` est le **produit matriciel** : NumPy fait `0.299·R + 0.587·G + 0.114·B` pixel par pixel. Les coefficients reflètent la sensibilité de l'œil : on est plus sensible au vert qu'au rouge ou au bleu. Le résultat est un **float** → recast :

```python
gray = (arr @ [0.299, 0.587, 0.114]).astype(np.uint8)
```

**Seuillage binaire**

```python
mask = (gray > 128).astype(np.uint8) * 255
```

`gray > 128` renvoie un tableau de booléens. Le `.astype(np.uint8)` les transforme en 0/1, puis `* 255` en 0/255 → visible en niveaux de gris.

⚠️ **Piège uint8 + opérations** : `arr * 2` peut **déborder** (255 × 2 → 254 en uint8 par enroulement modulo). Solutions : (a) passer en float, (b) `np.clip(arr.astype(np.int16) * 2, 0, 255).astype(np.uint8)`.

</details>

In [4]:
# TODO

## Exercice 4 — Image synthétique

Construisez deux images synthétiques en partant de tableaux NumPy :

1. Un **gradient horizontal** de 256×256 pixels, en niveaux de gris, dont l'intensité passe linéairement de 0 (gauche) à 255 (droite). Indice : `np.linspace` ou `np.tile`.
2. Un **échiquier** de 8×8 cases de 32×32 pixels chacune (image 256×256). Cases blanches (255) et noires (0) en alternance.

Affichez les deux images en niveaux de gris.

<details>
<summary>💡 Coup de pouce — synthèse d'images NumPy</summary>

**🎯 But :** créer des images **from scratch** pour bien comprendre comment NumPy organise les pixels.

**Gradient horizontal**

```python
ligne = np.linspace(0, 255, 256, dtype=np.uint8)   # 1D : [0, 1, 2, ..., 255]
gradient = np.tile(ligne, (256, 1))                # 2D : 256 copies de ligne
```

`np.tile(ligne, (256, 1))` réplique la même ligne **256 fois** verticalement. Le `(256, 1)` veut dire « 256× sur le 1er axe, 1× sur le 2e ». Résultat : gradient horizontal du noir au blanc.

**Échiquier**

Pour `n × n` cases noires/blanches de taille `c` :

```python
ix = np.arange(256) // 32           # 0,0,...,0,1,1,...,1,...,7,7,...,7
echiq = (ix[:, None] + ix[None, :]) % 2 * 255
```

Décomposition :
- `ix // 32` regroupe les pixels par bandes de 32 → indice de la case.
- `ix[:, None]` (colonne) + `ix[None, :]` (ligne) avec broadcast → carte 2D des sommes (ligne + colonne).
- `% 2` : alterne 0/1 sur le quadrillage.
- `* 255` : noir ou blanc.

**Pourquoi `[:, None]` et `[None, :]` ?**

C'est la syntaxe pour **forcer le broadcasting** entre un tableau ligne et un tableau colonne. C'est l'équivalent NumPy de `np.meshgrid` mais plus économe en mémoire.

</details>

In [5]:
# TODO

## Exercice 5 — Mosaïque 2×2

Composez une **mosaïque 2×2** combinant 4 versions de l'image `astronaut` :

- en haut à gauche : l'originale ;
- en haut à droite : son négatif ;
- en bas à gauche : sa version en niveaux de gris (répétée sur 3 canaux pour rester en RGB) ;
- en bas à droite : sa version où seul le canal rouge est conservé (les autres mis à zéro).

Le résultat est un tableau NumPy unique de shape `(H*2, W*2, 3)` et `dtype=uint8`. Affichez-le et sauvegardez-le via Pillow.

<details>
<summary>💡 Coup de pouce — assembler une mosaïque 2×2</summary>

**🎯 But :** composer une image finale à partir de 4 variantes en utilisant les fonctions d'assemblage NumPy.

**Méthode `np.block`**

Le plus lisible : on décrit la structure 2D avec une liste de listes, comme on écrirait une matrice :

```python
mosaique = np.block([[a, b],
                     [c, d]])
```

⚠️ **Contraintes** : `a` et `b` doivent avoir **la même hauteur** ; `a` et `c` la **même largeur**. Si on assemble du RGB, les 4 doivent avoir la même shape `(H, W, 3)` exactement.

**Astuce — passer du gris au RGB**

Si une de vos 4 sous-images est mono-canal (par exemple le résultat d'une luminance), il faut **dupliquer le canal** pour qu'elle ait 3 canaux comme les autres :

```python
gray3 = np.stack([gray, gray, gray], axis=-1)   # (H, W) → (H, W, 3)
```

`axis=-1` crée le nouvel axe **à la fin** (le canal). C'est l'équivalent NumPy de « peindre la même intensité en R, G et B ».

**Astuce — image avec seul canal rouge**

```python
red_only = arr.copy()
red_only[..., 1:] = 0   # canaux 1 et 2 (G et B) à 0, R intact
```

Le `1:` est un slice qui prend tous les canaux à partir du 1er → annule G et B en une ligne.

**Variantes** : `np.hstack` (côte à côte horizontalement) et `np.vstack` (empilage vertical) sont des cousins plus simples mais qui n'assemblent qu'**une dimension à la fois**.

</details>

In [6]:
# TODO